In [ ]:
import re, csv, io, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import MiniBatchKMeans
from sklearn.linear_model import LinearRegression

import matplotlib.pyplot as plt
from textblob import TextBlob

MODEL = "xgboost"
if MODEL == "catboost":
    from catboost import CatBoostClassifier, Pool

import xgboost as xgb

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
warnings.filterwarnings("ignore")

FAST_MODE = True
RUN_VARIANT_A = True
RUN_VARIANT_B = False if FAST_MODE else True

DATASET_PATH = "twitter_sentiments.csv"
RANDOM_STATE = 42
TEST_SIZE = 0.20

MAX_FEATURES_TFIDF = 50_000 if FAST_MODE else None
MIN_DF = 5 if FAST_MODE else 2
MAX_DF = 0.95

SVD_50 = 40 if FAST_MODE else 50
SVD_2 = 2
KMEANS_K = 3
MBATCH_SIZE = 1024

ROLL_WINDOW = 200 if FAST_MODE else 500
FORECAST_HORIZON = 100 if FAST_MODE else 200

EVAL_OUT_DIR = "analysis_outputs_fast"
Path(EVAL_OUT_DIR).mkdir(parents=True, exist_ok=True)

nltk.download("punkt", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
STOPWORDS = set(stopwords.words("english"))
LEMMATIZER = WordNetLemmatizer()

LIKELY_TEXT_COLS = {
    "text","tweet","content","review","message","comment","body","sentence",
    "sentimenttext","raw_tweet","tweet_text","clean_text"
}
LIKELY_LABEL_COLS = {
    "label","labels","sentiment","target","polarity","class","category","sentiment_label"
}

def _sniff_sep(sample: str, default=","):
    try:
        dialect = csv.Sniffer().sniff(sample, delimiters=",;\t|")
        return dialect.delimiter
    except Exception:
        return default

def _repair_csv_quotes(raw_lines):
    def _count_unescaped_quotes(s: str) -> int:
        return sum(1 for i,ch in enumerate(s) if ch == '"' and (i == 0 or s[i-1] != '\\'))
    fixed, buf, open_q = [], [], 0
    for line in raw_lines:
        l = line.rstrip("\r\n")
        q = _count_unescaped_quotes(l)
        buf.append(l); open_q += q
        if open_q % 2 == 0:
            fixed.append("".join([buf[0]] + [(" " + x) for x in buf[1:]]))
            buf, open_q = [], 0
    if buf:
        fixed.append("".join([buf[0]] + [(" " + x) for x in buf[1:]]) + '"')
    return fixed

def _read_with_best_guess(path: Path):
    with path.open("r", encoding="utf-8", errors="ignore") as f:
        head_lines = []
        for _ in range(50):
            try: head_lines.append(next(f))
            except StopIteration: break
        head = "".join(head_lines)
    sep = _sniff_sep(head, default=",")
    try:
        return pd.read_csv(path, sep=sep, engine="python", quoting=csv.QUOTE_MINIMAL,
                           escapechar="\\", encoding="utf-8")
    except Exception:
        try:
            df = pd.read_csv(path, sep=sep, engine="python", quoting=csv.QUOTE_MINIMAL,
                             escapechar="\\", encoding="utf-8", header=None)
            df.columns = [f"col_{i}" for i in range(df.shape[1])]
            return df
        except Exception:
            with path.open("r", encoding="utf-8", errors="ignore") as f:
                raw = f.readlines()
            fixed = _repair_csv_quotes(raw)
            sample = "".join(fixed[:50])
            sep2 = _sniff_sep(sample, default=sep)
            try:
                return pd.read_csv(io.StringIO("\n".join(fixed)), sep=sep2, engine="python",
                                   quoting=csv.QUOTE_MINIMAL, escapechar="\\", encoding="utf-8")
            except Exception:
                return pd.read_csv(io.StringIO("\n".join(fixed)), sep=sep2, engine="python",
                                   quoting=csv.QUOTE_MINIMAL, escapechar="\\", encoding="utf-8",
                                   on_bad_lines="skip")

def _pick_text_col(df: pd.DataFrame):
    for c in df.columns:
        if str(c).strip().lower() in LIKELY_TEXT_COLS:
            return c
    cand = [c for c in df.columns if df[c].dtype == "object"]
    if not cand:
        for c in df.columns: df[c] = df[c].astype(str)
        cand = list(df.columns)
    scores = {}
    for c in cand:
        s = df[c].astype(str).fillna("")
        if s.str.match(r"^\d+$").mean() > 0.8: continue
        scores[c] = s.str.len().mean()
    return max(scores, key=scores.get) if scores else None

def _pick_label_col(df: pd.DataFrame):
    for c in df.columns:
        if str(c).strip().lower() in LIKELY_LABEL_COLS:
            return c
    best, best_card = None, 999
    for c in df.columns:
        s = pd.Series(df[c]).dropna().astype(str).str.strip()
        card = s.nunique()
        if 2 <= card <= 5:
            if df[c].dtype == "object" and s.str.len().mean() > 30:
                continue
            if card < best_card:
                best, best_card = c, card
    return best

def load_dataset(path, text_override=None, label_override=None):
    path = Path(path)
    if not path.exists(): raise FileNotFoundError(f"CSV not found at {path.resolve()}")
    df = _read_with_best_guess(path)
    df.columns = [str(c).strip().replace("\ufeff","") for c in df.columns]
    text_col  = text_override  or _pick_text_col(df)
    label_col = label_override or _pick_label_col(df)
    if text_col is None or label_col is None:
        raise ValueError(
            "Could not find text/label columns.\n"
            f"Detected: {[repr(c) for c in df.columns]}\n"
            "Tip: load_dataset(path, text_override='clean_text', label_override='category')"
        )
    df = df[[text_col, label_col]].rename(columns={text_col: "text", label_col: "label"})
    df = df.dropna(subset=["text","label"]).reset_index(drop=True)
    return df

URL_PATTERN       = re.compile(r"https?://\S+|www\.\S+")
MENTION_PATTERN   = re.compile(r"@\w+")
HASHTAG_PATTERN   = re.compile(r"#\w+")
NON_ALPHA_PATTERN = re.compile(r"[^a-zA-Z\s]")

def clean_text_paper_style(text: str) -> str:
    if not isinstance(text, str): return ""
    t = text.lower()
    t = URL_PATTERN.sub(" ", t)
    t = MENTION_PATTERN.sub(" ", t)
    t = HASHTAG_PATTERN.sub(lambda m: m.group(0)[1:], t)
    t = NON_ALPHA_PATTERN.sub(" ", t)
    t = re.sub(r"\s+", " ", t).strip()
    tokens = [LEMMATIZER.lemmatize(tok) for tok in t.split() if tok not in STOPWORDS]
    return " ".join(tokens)

def coerce_to_m1_0_1(series: pd.Series) -> pd.Series:
    try:
        vals = series.astype(int)
        if set(vals.unique()) <= set([-1,0,1]): return vals
    except: pass
    for m in [
        {"negative":-1,"neutral":0,"positive":1},
        {"Negative":-1,"Neutral":0,"Positive":1},
        {"NEGATIVE":-1,"NEUTRAL":0,"POSITIVE":1},
        {"-1":-1,"0":0,"1":1},
        {"neg":-1,"neu":0,"pos":1},
        {"negative ": -1, " neutral":0, " positive":1},
    ]:
        mapped = series.astype(str).str.strip().map(m)
        if mapped.notna().all() and set(mapped.unique()) <= set([-1,0,1]):
            return mapped.astype(int)
    le = LabelEncoder()
    enc = le.fit_transform(series.astype(str))
    uniq = sorted(np.unique(enc))
    rank_map = {uniq[0]: -1, uniq[len(uniq)//2]: 0, uniq[-1]: 1} if len(uniq) == 3 else {u:i-1 for i,u in enumerate(uniq)}
    return pd.Series(enc).map(rank_map).astype(int)

def _get_proba_catboost(clf, X):
    try: return clf.predict_proba(X)
    except Exception: return None

def build_eval_dataframe(texts, y_true_orig, y_pred_orig, proba=None):
    df_eval = pd.DataFrame({
        "text": list(texts),
        "y_true": y_true_orig,
        "y_pred": y_pred_orig,
    })
    df_eval["correct"] = (df_eval["y_true"] == df_eval["y_pred"]).astype(int)
    df_eval["char_len"] = df_eval["text"].astype(str).str.len()
    df_eval["word_len"] = df_eval["text"].astype(str).str.split().map(len).fillna(0)

    if not FAST_MODE:
        pol, sub = [], []
        for t in df_eval["text"].astype(str):
            s = TextBlob(t).sentiment
            pol.append(s.polarity); sub.append(s.subjectivity)
        df_eval["tb_polarity"] = pol
        df_eval["tb_subjectivity"] = sub

    if proba is not None and hasattr(proba, "shape") and proba.shape[1] == 3:
        df_eval["p_neg"] = proba[:,0]; df_eval["p_neu"] = proba[:,1]; df_eval["p_pos"] = proba[:,2]
        df_eval["pred_confidence"] = proba.max(axis=1)
    else:
        df_eval["pred_confidence"] = np.nan
    return df_eval

def plot_corr_heatmap(df_eval, fname="corr_heatmap.png"):
    num_cols = df_eval.select_dtypes(include=[np.number]).columns
    corr = df_eval[num_cols].corr()
    plt.figure()
    im = plt.imshow(corr, aspect="auto")
    plt.xticks(range(len(num_cols)), num_cols, rotation=90)
    plt.yticks(range(len(num_cols)), num_cols)
    plt.title("Correlation Heatmap (engineered features)")
    plt.colorbar(im)
    plt.tight_layout()
    plt.savefig(Path(EVAL_OUT_DIR)/fname, dpi=200); plt.close()

def plot_trend_and_forecast(df_eval, window=ROLL_WINDOW, horizon=FORECAST_HORIZON,
                            fname_trend="trend.png", fname_fore="forecast.png"):
    roll = df_eval["correct"].rolling(window=window, min_periods=max(1, window//5)).mean()
    plt.figure(); plt.plot(roll.index, roll.values)
    plt.title(f"Rolling Accuracy (window={window})")
    plt.xlabel("Test sample index"); plt.ylabel("Accuracy")
    plt.tight_layout(); plt.savefig(Path(EVAL_OUT_DIR)/fname_trend, dpi=200); plt.close()

    series = roll.fillna(method="bfill").values
    alpha = 0.3; level = series[0]; smoothed = [level]
    for x in series[1:]:
        level = alpha*x + (1-alpha)*level; smoothed.append(level)
    last_level = smoothed[-1]
    forecast = np.full(horizon, last_level)
    plt.figure()
    plt.plot(range(len(series)), series, label="rolling_acc")
    plt.plot(range(len(smoothed)), smoothed, label="smoothed")
    plt.plot(range(len(series), len(series)+horizon), forecast, label="forecast")
    plt.legend(); plt.title("Exponential Smoothing Forecast of Rolling Accuracy")
    plt.xlabel("Index"); plt.ylabel("Accuracy")
    plt.tight_layout(); plt.savefig(Path(EVAL_OUT_DIR)/fname_fore, dpi=200); plt.close()

def cluster_and_plot(X_sparse, fname="clusters.png"):
    svd50 = TruncatedSVD(n_components=SVD_50, random_state=42)
    z50 = svd50.fit_transform(X_sparse)
    svd2 = TruncatedSVD(n_components=SVD_2, random_state=42)
    z2 = svd2.fit_transform(z50)
    km = MiniBatchKMeans(n_clusters=KMEANS_K, batch_size=MBATCH_SIZE, n_init=5, random_state=42)
    _ = km.fit_predict(z2)
    plt.figure(); plt.scatter(z2[:,0], z2[:,1], s=6, alpha=0.6)
    plt.title(f"MiniBatchKMeans (k={KMEANS_K}) on TF-IDF (2D SVD)")
    plt.xlabel("SVD-1"); plt.ylabel("SVD-2")
    plt.tight_layout(); plt.savefig(Path(EVAL_OUT_DIR)/fname, dpi=200); plt.close()
    return z2

def multiple_linear_regression_from_text(X_sparse, texts, fname="mlr_report.json"):
    svd = TruncatedSVD(n_components=SVD_50, random_state=42)
    Xs = svd.fit_transform(X_sparse)

    out = {"note": "FAST mode: skipping heavy polarity regression",
           "svd_explained_variance_ratio_sum": float(svd.explained_variance_ratio_.sum())}
    with open(Path(EVAL_OUT_DIR)/fname, "w") as f: json.dump(out, f, indent=2)
    return out

def plot_loss_and_accuracy_curves(model, eval_name="validation",
                                  fname_loss="loss_curve.png", fname_acc="acc_curve.png"):
    """
    Supports:
      - XGBoost Booster (native): looks for model._evals_result (dict from train)
      - CatBoost: uses get_evals_result()
    """

    try:
        res = getattr(model, "_evals_result", None)
        if isinstance(res, dict) and eval_name in res:
            if "mlogloss" in res[eval_name]:
                plt.figure(); plt.plot(res[eval_name]["mlogloss"])
                plt.title("Validation mlogloss"); plt.xlabel("Iteration"); plt.ylabel("mlogloss")
                plt.tight_layout(); plt.savefig(Path(EVAL_OUT_DIR)/fname_loss, dpi=200); plt.close()
            if "merror" in res[eval_name]:
                acc = 1 - np.array(res[eval_name]["merror"], dtype=float)
                plt.figure(); plt.plot(acc)
                plt.title("Validation Accuracy (1 - merror)"); plt.xlabel("Iteration"); plt.ylabel("Accuracy")
                plt.tight_layout(); plt.savefig(Path(EVAL_OUT_DIR)/fname_acc, dpi=200); plt.close()
        return
    except Exception:
        pass

    try:
        res = model.get_evals_result()
        val_dict = res.get("validation", {})
        if "MultiClass" in val_dict:
            plt.figure(); plt.plot(val_dict["MultiClass"])
            plt.title("CatBoost Validation MultiClass Loss"); plt.xlabel("Iteration"); plt.ylabel("Loss")
            plt.tight_layout(); plt.savefig(Path(EVAL_OUT_DIR)/fname_loss, dpi=200); plt.close()
        if "TotalF1" in val_dict:
            plt.figure(); plt.plot(val_dict["TotalF1"])
            plt.title("CatBoost Validation TotalF1"); plt.xlabel("Iteration"); plt.ylabel("TotalF1")
            plt.tight_layout(); plt.savefig(Path(EVAL_OUT_DIR)/fname_acc, dpi=200); plt.close()
    except Exception:
        return

def train_eval_model(text_series, label_series, model_kind=MODEL):
    classes_sorted = sorted(pd.Series(label_series).unique().tolist())
    to_zero = {c: i for i, c in enumerate(classes_sorted)}
    to_orig = {i: c for c, i in to_zero.items()}
    y0 = pd.Series(label_series).map(to_zero).astype(int)

    X_train_all, X_test, y_train_all, y_test0, y_train_all_orig, y_test_orig = train_test_split(
        pd.Series(text_series).astype(str),
        y0,
        pd.Series(label_series),
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=y0
    )
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train_all, y_train_all, test_size=0.1, random_state=RANDOM_STATE, stratify=y_train_all
    )

    vectorizer = TfidfVectorizer(
        ngram_range=(1,1),
        max_df=MAX_DF,
        min_df=MIN_DF,
        max_features=MAX_FEATURES_TFIDF
    )
    Xtr = vectorizer.fit_transform(X_tr)
    Xval = vectorizer.transform(X_val)
    Xte  = vectorizer.transform(X_test)

    if model_kind == "xgboost":
        dtrain = xgb.DMatrix(Xtr, label=y_tr)
        dvalid = xgb.DMatrix(Xval, label=y_val)
        dtest  = xgb.DMatrix(Xte)

        params = {
            "objective": "multi:softprob",
            "num_class": len(classes_sorted),
            "eta": 0.06,
            "max_depth": 8,
            "subsample": 0.9,
            "colsample_bytree": 0.9,
            "reg_lambda": 1.0,
            "tree_method": "hist",
            "eval_metric": ["mlogloss", "merror"],
            "seed": RANDOM_STATE,
            "nthread": -1
        }

        evals_result = {}
        booster = xgb.train(
            params,
            dtrain,
            num_boost_round=2000,
            evals=[(dvalid, "validation")],
            early_stopping_rounds=50,
            verbose_eval=False,
            evals_result=evals_result
        )

        booster._evals_result = evals_result

        proba = booster.predict(dtest)
        pred0 = proba.argmax(axis=1).astype(int)
        clf = booster

    elif model_kind == "catboost":
        train_pool = Pool(Xtr, y_tr)
        valid_pool = Pool(Xval, y_val)
        clf = CatBoostClassifier(
            loss_function="MultiClass",
            eval_metric="TotalF1",
            learning_rate=0.06,
            depth=8,
            iterations=2000,
            random_seed=RANDOM_STATE,
            verbose=False,
            od_type="Iter",
            od_wait=50
        )
        clf.fit(train_pool, eval_set=valid_pool, use_best_model=True)
        proba = _get_proba_catboost(clf, Xte)
        pred0 = np.array(clf.predict(Xte)).reshape(-1).astype(int)

    else:
        raise ValueError("MODEL must be 'xgboost' or 'catboost'.")

    y_pred_orig = np.array([to_orig[i] for i in pred0])
    y_true_orig = np.array(y_test_orig)

    acc = accuracy_score(y_true_orig, y_pred_orig)
    f1m = f1_score(y_true_orig, y_pred_orig, average="macro")
    print(f"Accuracy: {acc:.4f} | Macro-F1: {f1m:.4f}")
    print("Confusion matrix:\n", confusion_matrix(y_true_orig, y_pred_orig, labels=classes_sorted))
    print(classification_report(y_true_orig, y_pred_orig, digits=4, labels=classes_sorted))

    rpt_dict = classification_report(y_true_orig, y_pred_orig, output_dict=True, digits=4,
                                     labels=classes_sorted, zero_division=0)
    pd.DataFrame(rpt_dict).transpose().to_csv(Path(EVAL_OUT_DIR)/f"report_{model_kind}.csv")
    cm = confusion_matrix(y_true_orig, y_pred_orig, labels=classes_sorted)
    pd.DataFrame(cm, index=[f"true_{c}" for c in classes_sorted],
                    columns=[f"pred_{c}" for c in classes_sorted]).to_csv(
        Path(EVAL_OUT_DIR)/f"confusion_matrix_{model_kind}.csv"
    )

    df_eval = build_eval_dataframe(list(X_test), y_true_orig, y_pred_orig, proba=proba)
    df_eval.to_csv(Path(EVAL_OUT_DIR)/f"eval_rows_{model_kind}.csv", index=False)

    plot_corr_heatmap(df_eval, fname=f"corr_heatmap_{model_kind}.png")
    plot_trend_and_forecast(df_eval, window=ROLL_WINDOW, horizon=FORECAST_HORIZON,
                            fname_trend=f"trend_{model_kind}.png",
                            fname_fore=f"forecast_{model_kind}.png")
    cluster_and_plot(Xte, fname=f"clusters_{model_kind}.png")
    multiple_linear_regression_from_text(Xte, list(X_test), fname=f"mlr_{model_kind}.json")
    plot_loss_and_accuracy_curves(clf, eval_name="validation",
                                  fname_loss=f"loss_curve_{model_kind}.png",
                                  fname_acc=f"acc_curve_{model_kind}.png")
    return acc, f1m

def run_variant_A(df):
    if "clean_text" in df.columns and df["clean_text"].notna().any():
        texts = df["clean_text"].astype(str)
    else:
        texts = df["text"].astype(str).apply(clean_text_paper_style)
    labels = coerce_to_m1_0_1(df["label"])
    print("\n=== Variant A (Default labels; TF-IDF + GBM) ===")
    return train_eval_model(texts, labels)

def textblob_label(text: str) -> int:
    if not isinstance(text, str) or not text.strip(): return 0
    pol = TextBlob(text).sentiment.polarity
    if pol > 0: return 1
    if pol < 0: return -1
    return 0

def run_variant_B(df):
    if "clean_text" in df.columns and df["clean_text"].notna().any():
        raw = df["text"].astype(str) if "text" in df.columns else df["clean_text"].astype(str)
        cleaned = df["clean_text"].astype(str)
    else:
        raw = df["text"].astype(str)
        cleaned = raw.apply(clean_text_paper_style)
    tb_labels = raw.apply(textblob_label)
    if tb_labels.nunique() < 3:
        print("\n[WARN] TextBlob produced <3 classes; stratification may be weak.")
    print("\n=== Variant B (TextBlob re-annotation; TF-IDF + GBM) ===")
    return train_eval_model(cleaned, tb_labels)

if __name__ == "__main__":
    df = load_dataset(DATASET_PATH)
    if "text" not in df.columns and "clean_text" in df.columns:
        df = df.rename(columns={"clean_text": "text"})

    results = {}
    if RUN_VARIANT_A:
        accA, f1A = run_variant_A(df); results["variant_A"] = {"accuracy": float(accA), "macro_f1": float(f1A)}
    if RUN_VARIANT_B:
        accB, f1B = run_variant_B(df); results["variant_B"] = {"accuracy": float(accB), "macro_f1": float(f1B)}

    results["n_samples"] = int(df.shape[0])
    print("\nSUMMARY_JSON:", json.dumps(results, indent=2))
    print(f"\nArtifacts saved to: {Path(EVAL_OUT_DIR).resolve()}")



=== Variant A (Default labels; TF-IDF + GBM) ===


In [ ]:
!pip -q install xgboost pandas numpy scikit-learn matplotlib

import io, csv, re, json, warnings
from pathlib import Path
from typing import Tuple, List, Dict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, f1_score, confusion_matrix, classification_report, cohen_kappa_score
)
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
import xgboost as xgb

warnings.filterwarnings("ignore")

try:
    from google.colab import files
    up = files.upload()
    DATASET_PATH = list(up.keys())[0]
except Exception:
    DATASET_PATH = "/content/data.csv"

RANDOM_STATE = 42
TEST_SIZE    = 0.20
VAL_SIZE     = 0.10
MAX_FEATURES_TFIDF = 200_000
MIN_DF = 3
MAX_DF = 0.98
USE_XGB = True

OUT_DIR = Path("analysis_outputs_full"); OUT_DIR.mkdir(exist_ok=True)

LIKELY_TEXT  = {'text','tweet','review','content','message','comment','body','sentence','clean_text'}
LIKELY_LABEL = {'label','labels','sentiment','target','polarity','class','category','sentiment_label'}

def _pick_col(df, cands):
    for c in df.columns:
        if str(c).strip().lower() in cands:
            return c
    return None

def load_dataset(path)->pd.DataFrame:
    df = pd.read_csv(path)
    df.columns = [str(c).strip().replace("\ufeff","") for c in df.columns]
    tcol = _pick_col(df, LIKELY_TEXT)  or df.columns[0]
    lcol = _pick_col(df, LIKELY_LABEL) or df.columns[1]
    df = df[[tcol, lcol]].rename(columns={tcol:"text", lcol:"label"})
    df = df.dropna(subset=["text","label"]).reset_index(drop=True)
    return df

URL = re.compile(r"https?://\S+|www\.\S+")
MENTION = re.compile(r"@\w+")
MULTI_WS = re.compile(r"\s+")

def clean_text(s:str)->str:
    if not isinstance(s,str): return ""
    s = s.lower()
    s = URL.sub(" ", s)
    s = MENTION.sub(" ", s)
    s = s.replace("#","")
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = MULTI_WS.sub(" ", s).strip()
    return s

def coerce_labels(y:pd.Series)->Tuple[np.ndarray, Dict[int, str]]:
    """Return y_int (0..K-1) and index->class_name map."""
    maps = [
        {"negative":-1,"neutral":0,"positive":1},
        {"neg":-1,"neu":0,"pos":1},
        {"-1":-1,"0":0,"1":1},
        {"NEGATIVE":-1,"NEUTRAL":0,"POSITIVE":1},
        {"Negative":-1,"Neutral":0,"Positive":1},
    ]
    y_str = y.astype(str).str.strip()
    for m in maps:
        mapp = y_str.map(m)
        if mapp.notna().mean() > 0.9 and set(mapp.dropna().unique()) <= {-1,0,1}:
            yy = mapp.fillna(0).astype(int)
            le = LabelEncoder()
            enc = le.fit_transform(yy)
            inv = {i: v for i,v in enumerate(le.inverse_transform(sorted(set(enc))))}
            return enc.astype(int), {i: str(inv[i]) for i in inv}
    le = LabelEncoder()
    enc = le.fit_transform(y_str)
    inv = {i: c for i,c in enumerate(le.inverse_transform(np.arange(enc.max()+1)))}
    return enc.astype(int), inv

def bar_plot(values:Dict[str,float], title:str, fname:str):
    names = list(values.keys()); vals = list(values.values())
    plt.figure()
    plt.bar(range(len(vals)), vals)
    plt.xticks(range(len(vals)), names, rotation=45, ha='right')
    plt.title(title); plt.tight_layout()
    plt.savefig(OUT_DIR/fname, dpi=220); plt.close()

def plot_confusion(cm:np.ndarray, class_names:List[str], title:str, fname:str):
    plt.figure()
    im = plt.imshow(cm, cmap="Blues")
    plt.title(title); plt.colorbar(im)
    plt.xticks(range(len(class_names)), class_names, rotation=45, ha='right')
    plt.yticks(range(len(class_names)), class_names)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, str(cm[i, j]), ha="center", va="center")
    plt.tight_layout()
    plt.savefig(OUT_DIR/fname, dpi=240); plt.close()

def per_class_linear_regression_plots(y_true_int:np.ndarray, proba:np.ndarray, class_names:List[str]):
    """
    For each class k: regress 1[y==k] ~ proba[:,k] ; plot scatter + fitted line
    """
    for k, cname in enumerate(class_names):
        yk = (y_true_int == k).astype(float)
        pk = proba[:, k]
        X = pk.reshape(-1,1)
        lr = LinearRegression().fit(X, yk)
        x_line = np.linspace(pk.min(), pk.max(), 100).reshape(-1,1)
        y_line = lr.predict(x_line)

        plt.figure()
        plt.scatter(pk, yk, s=8, alpha=0.3)
        plt.plot(x_line, y_line)
        plt.title(f"Linear Regression: 1[y={cname}] vs P(y={cname})  |  R²={lr.score(X, yk):.3f}")
        plt.xlabel(f"P(y={cname})"); plt.ylabel(f"Indicator(y={cname})")
        plt.tight_layout()
        plt.savefig(OUT_DIR/f"linreg_{k}_{cname}.png", dpi=220); plt.close()

df = load_dataset(DATASET_PATH)
df["text"] = df["text"].astype(str).map(clean_text)

y_int, inv_map = coerce_labels(df["label"])
class_names = [inv_map[i] for i in sorted(inv_map.keys())]
K = len(class_names)

cls_counts = df["label"].astype(str).value_counts().to_dict()
bar_plot(cls_counts, "Class Distribution", "bar_class_distribution.png")

X_train_all, X_test, y_train_all, y_test = train_test_split(
    df["text"], y_int,
    test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_int
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_all, y_train_all,
    test_size=VAL_SIZE, random_state=RANDOM_STATE, stratify=y_train_all
)

tfidf = TfidfVectorizer(
    ngram_range=(1,2),
    sublinear_tf=True,
    min_df=MIN_DF,
    max_df=MAX_DF,
    max_features=MAX_FEATURES_TFIDF
)

Xtr = tfidf.fit_transform(X_train)
Xv  = tfidf.transform(X_val)
Xte = tfidf.transform(X_test)

svc = LinearSVC(C=2.5, class_weight="balanced", max_iter=5000, random_state=RANDOM_STATE)
svc_cal = CalibratedClassifierCV(svc, method="isotonic", cv=3)
svc_cal.fit(Xtr, y_train)

logit = LogisticRegression(
    C=6.0, class_weight="balanced", max_iter=5000, n_jobs=-1, solver="saga", penalty="l2"
)
logit.fit(Xtr, y_train)

if USE_XGB:
    dtr = xgb.DMatrix(Xtr, label=y_train)
    dv  = xgb.DMatrix(Xv,  label=y_val)
    params = {
        "objective": "multi:softprob",
        "num_class": K,
        "eta": 0.08,
        "max_depth": 8,
        "subsample": 0.9,
        "colsample_bytree": 0.9,
        "reg_lambda": 1.5,
        "tree_method": "hist",
        "eval_metric": ["mlogloss","merror"],
        "seed": RANDOM_STATE,
    }
    ev = {}
    xgb_model = xgb.train(
        params, dtr, num_boost_round=1500,
        evals=[(dv,"val")], early_stopping_rounds=75, verbose_eval=False, evals_result=ev
    )
    xgb_model._evals_result = ev
else:
    xgb_model = None

def proba(model, X):
    if hasattr(model, "predict_proba"): return model.predict_proba(X)
    if isinstance(model, xgb.Booster): return model.predict(xgb.DMatrix(X))
    raise ValueError("Model has no probability interface")

probs_val = []
names = []
p_svc = proba(svc_cal, Xv);  probs_val.append(p_svc); names.append("svc")
p_log = proba(logit, Xv);    probs_val.append(p_log); names.append("logit")
if xgb_model is not None:
    p_xgb = proba(xgb_model, Xv); probs_val.append(p_xgb); names.append("xgb")

weights = np.ones(len(probs_val)) / len(probs_val)
best_f1, best_w = -1.0, weights.copy()

def search_weights(probs_list, y_true, steps=(14, 14, 14)):
    global best_f1, best_w
    grids = [np.linspace(0.2, 1.6, steps[i]) for i in range(len(probs_list))]
    if len(probs_list) == 2:
        for a in grids[0]:
            for b in grids[1]:
                w = np.array([a,b])
                mix = np.average(np.stack(probs_list, axis=0), axis=0, weights=w)
                f1v = f1_score(y_true, mix.argmax(axis=1), average="macro")
                if f1v > best_f1: best_f1, best_w = f1v, w
    else:
        for a in grids[0]:
            for b in grids[1]:
                for c in grids[2]:
                    w = np.array([a,b,c])
                    mix = np.average(np.stack(probs_list, axis=0), axis=0, weights=w)
                    f1v = f1_score(y_true, mix.argmax(axis=1), average="macro")
                    if f1v > best_f1: best_f1, best_w = f1v, w

search_weights(probs_val, y_val)
print(f"Chosen weights (validation macro-F1={best_f1:.4f}):", dict(zip(names, [float(x) for x in best_w])))

p_svc_te = proba(svc_cal, Xte)
p_log_te = proba(logit, Xte)
prob_stack = [p_svc_te, p_log_te]
if xgb_model is not None:
    p_xgb_te = proba(xgb_model, Xte)
    prob_stack.append(p_xgb_te)

P = np.average(np.stack(prob_stack, axis=0), axis=0, weights=best_w)
y_pred = P.argmax(axis=1)

acc   = accuracy_score(y_test, y_pred)
f1m   = f1_score(y_test, y_pred, average="macro")
kappa = cohen_kappa_score(y_test, y_pred)

print("\n=== ENSEMBLE RESULTS (Test) ===")
print(f"Accuracy     : {acc:.4f}")
print(f"Macro-F1     : {f1m:.4f}")
print(f"Cohen's kappa: {kappa:.4f}\n")
print(classification_report(y_test, y_pred, digits=4, target_names=class_names))

cm = confusion_matrix(y_test, y_pred)
plot_confusion(cm, class_names, "Confusion Matrix (Ensemble)", "confusion_matrix.png")

report = classification_report(y_test, y_pred, output_dict=True, digits=4, target_names=class_names)
per_class_f1 = {cls: report[cls]["f1-score"] for cls in class_names}
bar_plot(per_class_f1, "Per-class F1 (Ensemble)", "bar_per_class_f1.png")

per_class_linear_regression_plots(y_test, P, class_names)

summary = {
    "accuracy": float(acc),
    "macro_f1": float(f1m),
    "cohen_kappa": float(kappa),
    "class_f1": {k: float(v) for k, v in per_class_f1.items()},
    "weights": dict(zip(names, [float(w) for w in best_w])),
    "n_train": int(len(X_train_all)),
    "n_test": int(len(X_test)),
    "classes": class_names
}
with open(OUT_DIR/"summary.json","w") as f: json.dump(summary, f, indent=2)

print(f"\nArtifacts saved to: {OUT_DIR.resolve()}")
for p in sorted(OUT_DIR.glob("*")): print(" -", p.name)

In [ ]:
!pip -q install xgboost pandas numpy scikit-learn matplotlib

import re, json, warnings, numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from typing import Dict, List, Tuple
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import MiniBatchKMeans
import xgboost as xgb

warnings.filterwarnings("ignore")

try:
    from google.colab import files
    up = files.upload()
    DATASET_PATH = list(up.keys())[0]
except Exception:
    DATASET_PATH = "/content/data.csv"

RANDOM_STATE = 42
TEST_SIZE    = 0.20
VAL_SIZE     = 0.10
MAX_FEATURES_TFIDF = 200_000
MIN_DF = 3
MAX_DF = 0.98
USE_XGB = True

OUT_DIR = Path("plots_from_run")
OUT_DIR.mkdir(exist_ok=True)

LIKELY_TEXT  = {'text','tweet','review','content','message','comment','body','sentence','clean_text'}
LIKELY_LABEL = {'label','labels','sentiment','target','polarity','class','category','sentiment_label'}

def _pick_col(df, cands):
    for c in df.columns:
        if str(c).strip().lower() in cands:
            return c
    return None

def load_dataset(path)->pd.DataFrame:
    df = pd.read_csv(path)
    df.columns = [str(c).strip().replace("\ufeff","") for c in df.columns]
    tcol = _pick_col(df, LIKELY_TEXT)  or df.columns[0]
    lcol = _pick_col(df, LIKELY_LABEL) or df.columns[1]
    df = df[[tcol, lcol]].rename(columns={tcol:"text", lcol:"label"})
    return df.dropna(subset=["text","label"]).reset_index(drop=True)

URL = re.compile(r"https?://\S+|www\.\S+")
MENTION = re.compile(r"@\w+")
MULTI_WS = re.compile(r"\s+")

def clean_text(s:str)->str:
    if not isinstance(s,str): return ""
    s = s.lower()
    s = URL.sub(" ", s)
    s = MENTION.sub(" ", s)
    s = s.replace("#","")
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = MULTI_WS.sub(" ", s).strip()
    return s

def coerce_labels(y:pd.Series)->Tuple[np.ndarray, List[str]]:
    maps = [
        {"negative":-1,"neutral":0,"positive":1},
        {"neg":-1,"neu":0,"pos":1},
        {"-1":-1,"0":0,"1":1},
        {"NEGATIVE":-1,"NEUTRAL":0,"POSITIVE":1},
        {"Negative":-1,"Neutral":0,"Positive":1},
    ]
    ys = y.astype(str).str.strip()
    for m in maps:
        mm = ys.map(m)
        if mm.notna().mean() > 0.9 and set(mm.dropna().unique()) <= {-1,0,1}:
            yy = mm.fillna(0).astype(int)
            le = LabelEncoder()
            enc = le.fit_transform(yy)
            classes = list(le.inverse_transform(sorted(set(enc))))
            return enc.astype(int), [str(c) for c in classes]
    le = LabelEncoder()
    enc = le.fit_transform(ys)
    classes = list(le.inverse_transform(np.arange(enc.max()+1)))
    return enc.astype(int), [str(c) for c in classes]

df = load_dataset(DATASET_PATH)
df["text"] = df["text"].astype(str).map(clean_text)
y_int, class_names = coerce_labels(df["label"])
K = len(class_names)

X_train_all, X_test, y_train_all, y_test = train_test_split(
    df["text"], y_int, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_int
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_all, y_train_all, test_size=VAL_SIZE, random_state=RANDOM_STATE, stratify=y_train_all
)

tfidf = TfidfVectorizer(
    ngram_range=(1,2), sublinear_tf=True,
    min_df=MIN_DF, max_df=MAX_DF, max_features=MAX_FEATURES_TFIDF
)

Xtr = tfidf.fit_transform(X_train)
Xv  = tfidf.transform(X_val)
Xte = tfidf.transform(X_test)

svc = LinearSVC(C=2.5, class_weight="balanced", max_iter=5000, random_state=RANDOM_STATE)
svc_cal = CalibratedClassifierCV(svc, method="isotonic", cv=3).fit(Xtr, y_train)

logit = LogisticRegression(C=6.0, class_weight="balanced", max_iter=5000,
                           n_jobs=-1, solver="saga", penalty="l2").fit(Xtr, y_train)

def proba(model, X):
    if hasattr(model, "predict_proba"): return model.predict_proba(X)
    if isinstance(model, xgb.Booster):  return model.predict(xgb.DMatrix(X))
    raise ValueError("No probability interface")

if USE_XGB:
    dtr = xgb.DMatrix(Xtr, label=y_train)
    dv  = xgb.DMatrix(Xv,  label=y_val)

    params = {
        "objective":"multi:softprob","num_class":K,"eta":0.08,"max_depth":8,
        "subsample":0.9,"colsample_bytree":0.9,"reg_lambda":1.5,
        "tree_method":"hist","eval_metric":["mlogloss","merror"],"seed":RANDOM_STATE
    }

    ev = {}
    xgb_model = xgb.train(params, dtr, num_boost_round=1200,
                          evals=[(dv,"val")], early_stopping_rounds=60,
                          verbose_eval=False, evals_result=ev)
    xgb_model._evals_result = ev
else:
    xgb_model = None

def best_weights(p_list, y_true):
    grids = [np.linspace(0.2, 1.6, 12) for _ in range(len(p_list))]
    best_f1, best_w = -1, np.ones(len(p_list))/len(p_list)
    if len(p_list)==2:
        for a in grids[0]:
            for b in grids[1]:
                w = np.array([a,b])
                mix = np.average(np.stack(p_list), axis=0, weights=w)
                f1 = f1_score(y_true, mix.argmax(1), average="macro")
                if f1>best_f1: best_f1, best_w = f1, w
    else:
        for a in grids[0]:
            for b in grids[1]:
                for c in grids[2]:
                    w = np.array([a,b,c])
                    mix = np.average(np.stack(p_list), axis=0, weights=w)
                    f1 = f1_score(y_true, mix.argmax(1), average="macro")
                    if f1>best_f1: best_f1, best_w = f1, w
    return best_w

val_probs = [proba(svc_cal, Xv), proba(logit, Xv)]
names = ["svc","logit"]
if xgb_model is not None:
    val_probs.append(proba(xgb_model, Xv)); names.append("xgb")

W = best_weights(val_probs, y_val)
print("Ensemble weights (val):", dict(zip(names, [float(x) for x in W])))

test_probs = [proba(svc_cal, Xte), proba(logit, Xte)]
if xgb_model is not None: test_probs.append(proba(xgb_model, Xte))
P = np.average(np.stack(test_probs), axis=0, weights=W)
y_pred = P.argmax(1)

print("Test accuracy:", accuracy_score(y_test, y_pred))
print("Test macro-F1:", f1_score(y_test, y_pred, average="macro"))

eval_df = pd.DataFrame({
    "text": X_test.values,
    "y_true": y_test,
    "y_pred": y_pred,
    "p_max": P.max(axis=1),
    "char_len": X_test.astype(str).str.len(),
    "word_len": X_test.astype(str).str.split().map(len).fillna(0).astype(int),
})

for k, cname in enumerate(class_names):
    eval_df[f"p_{cname}"] = P[:,k]
eval_df["correct"] = (eval_df["y_true"] == eval_df["y_pred"]).astype(int)


In [ ]:
import re, json, warnings, time
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from typing import List, Tuple

from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import MiniBatchKMeans

import xgboost as xgb
warnings.filterwarnings("ignore")

try:
    from google.colab import files
    print("[📁] Use the dialog to upload your CSV (must have text + label columns)…")
    up = files.upload()
    DATASET_PATH = list(up.keys())[0]
    print(f"[✔] Uploaded: {DATASET_PATH}")
except Exception:
    DATASET_PATH = "/content/data.csv"
    print(f"[ℹ] Using default path: {DATASET_PATH}")

RANDOM_STATE = 42
TEST_SIZE    = 0.20
VAL_SIZE     = 0.10

MAX_FEATURES_TFIDF = 60_000
MIN_DF = 5
MAX_DF = 0.95
NGRAMS = (1, 1)

USE_XGB      = True
USE_XGB_GPU  = True
USE_SVC      = False

XGB_MAX_ROUNDS    = 600
XGB_EARLY_STOP    = 20
XGB_LEARNING_RATE = 0.12
XGB_MAX_DEPTH     = 6
XGB_TIME_LIMIT_S  = None

RUN_PLOTS   = False
OUT_DIR     = Path("fast_run_artifacts")
OUT_DIR.mkdir(exist_ok=True)

def now(): return datetime.now().strftime("%H:%M:%S")
def log(msg): print(f"[{now()}] {msg}")

LIKELY_TEXT  = {'text','tweet','review','content','message','comment','body','sentence','clean_text'}
LIKELY_LABEL = {'label','labels','sentiment','target','polarity','class','category','sentiment_label'}

def _pick_col(df, cands):
    for c in df.columns:
        if str(c).strip().lower() in cands:
            return c
    return None

def load_dataset(path)->pd.DataFrame:
    t0 = time.time()
    log("Loading dataset…")
    df = pd.read_csv(path)
    df.columns = [str(c).strip().replace("\ufeff","") for c in df.columns]
    tcol = _pick_col(df, LIKELY_TEXT)  or df.columns[0]
    lcol = _pick_col(df, LIKELY_LABEL) or df.columns[1]
    df = df[[tcol, lcol]].rename(columns={tcol:"text", lcol:"label"})
    df = df.dropna(subset=["text","label"]).reset_index(drop=True)
    log(f"Loaded {len(df):,} rows (text='{tcol}', label='{lcol}') in {time.time()-t0:.2f}s")
    return df

URL = re.compile(r"https?://\S+|www\.\S+")
MENTION = re.compile(r"@\w+")
MULTI_WS = re.compile(r"\s+")

def _clean_one(s:str)->str:
    if not isinstance(s,str): return ""
    s = s.lower()
    s = URL.sub(" ", s)
    s = MENTION.sub(" ", s)
    s = s.replace("#","")
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = MULTI_WS.sub(" ", s).strip()
    return s

def clean_series(series: pd.Series, desc="Cleaning text")->pd.Series:
    log(f"{desc} (tqdm)…")
    t0 = time.time()
    tqdm.pandas(mininterval=0.5)
    out = series.progress_map(_clean_one)
    log(f"Done cleaning in {time.time()-t0:.2f}s")
    return out

def coerce_labels(y:pd.Series)->Tuple[np.ndarray, List[str]]:
    maps = [
        {"negative":-1,"neutral":0,"positive":1},
        {"neg":-1,"neu":0,"pos":1},
        {"-1":-1,"0":0,"1":1},
        {"NEGATIVE":-1,"NEUTRAL":0,"POSITIVE":1},
        {"Negative":-1,"Neutral":0,"Positive":1},
    ]
    ys = y.astype(str).str.strip()
    for m in maps:
        mm = ys.map(m)
        if mm.notna().mean() > 0.9 and set(mm.dropna().unique()) <= {-1,0,1}:
            yy = mm.fillna(0).astype(int)
            le = LabelEncoder(); enc = le.fit_transform(yy)
            classes = list(le.inverse_transform(sorted(set(enc))))
            return enc.astype(int), [str(c) for c in classes]
    le = LabelEncoder(); enc = le.fit_transform(ys)
    classes = list(le.inverse_transform(np.arange(enc.max()+1)))
    return enc.astype(int), [str(c) for c in classes]

def proba(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)
    if isinstance(model, xgb.Booster):
        return model.predict(xgb.DMatrix(X))
    raise ValueError("No probability interface")

def best_weights(p_list: List[np.ndarray], y_true: np.ndarray):
    log("Searching ensemble weights on validation…")
    grids = [np.linspace(0.5, 1.5, 5) for _ in range(len(p_list))]
    best_f1, best_w = -1, np.ones(len(p_list))/len(p_list)
    tried = 0
    if len(p_list)==2:
        for a in grids[0]:
            for b in grids[1]:
                w = np.array([a,b]); tried += 1
                mix = np.average(np.stack(p_list), axis=0, weights=w)
                f1 = f1_score(y_true, mix.argmax(1), average="macro")
                if f1>best_f1: best_f1, best_w = f1, w
    else:
        for a in grids[0]:
            for b in grids[1]:
                for c in grids[2]:
                    w = np.array([a,b,c]); tried += 1
                    mix = np.average(np.stack(p_list), axis=0, weights=w)
                    f1 = f1_score(y_true, mix.argmax(1), average="macro")
                    if f1>best_f1: best_f1, best_w = f1, w
    log(f"Grid tried={tried} combos; best macro-F1={best_f1:.4f}")
    return best_w

class TimeLimitCallback(xgb.callback.TrainingCallback):
    def __init__(self, seconds=None):
        self.seconds = seconds
        self.t0 = None
    def before_training(self, model):
        if self.seconds: import time; self.t0 = time.time()
        return model
    def after_iteration(self, model, epoch, evals_log):
        if not self.seconds: return False
        import time
        if time.time() - self.t0 > self.seconds:
            print(f"[⏱] Stopping early at iter {epoch} due to time limit ({self.seconds}s).")
            return True
        return False

def train_xgb_with_fallback(Xtr, y_train, Xv, y_val, K,
                            use_gpu=USE_XGB_GPU,
                            max_rounds=XGB_MAX_ROUNDS,
                            early_stop=XGB_EARLY_STOP,
                            lr=XGB_LEARNING_RATE,
                            depth=XGB_MAX_DEPTH,
                            time_limit_s=XGB_TIME_LIMIT_S):
    def _train_one(is_gpu: bool):
        dtr = xgb.DMatrix(Xtr, label=y_train)
        dv  = xgb.DMatrix(Xv,  label=y_val)
        params = {
            "objective": "multi:softprob",
            "num_class": K,
            "eta": lr,
            "max_depth": depth,
            "subsample": 0.8,
            "colsample_bytree": 0.8,
            "reg_lambda": 1.2,
            "eval_metric": ["mlogloss", "merror"],
            "seed": RANDOM_STATE,
            "tree_method": "gpu_hist" if is_gpu else "hist",
            "predictor": "gpu_predictor" if is_gpu else "auto",
            "nthread": -1,
        }
        ev = {}
        callbacks = []
        if time_limit_s is not None:
            callbacks.append(TimeLimitCallback(time_limit_s))
        log(f"Fitting XGBoost on {'GPU' if is_gpu else 'CPU'} (fast schedule, early stop)…")
        booster = xgb.train(
            params, dtr, num_boost_round=max_rounds,
            evals=[(dv, "val")],
            early_stopping_rounds=early_stop,
            verbose_eval=25,
            evals_result=ev,
            callbacks=callbacks
        )
        booster._evals_result = ev
        log(f"XGBoost best_iteration={booster.best_iteration} on {'GPU' if is_gpu else 'CPU'}")
        return booster

    if use_gpu:
        try:
            return _train_one(is_gpu=True)
        except xgb.core.XGBoostError as e:
            msg = str(e)
            log(f"[WARN] GPU training failed: {msg.strip().splitlines()[-1]}")
            log("Retrying on CPU ('hist')…")
            return _train_one(is_gpu=False)
    else:
        return _train_one(is_gpu=False)

df = load_dataset(DATASET_PATH)
df["text"] = clean_series(df["text"])

y_int, class_names = coerce_labels(df["label"])
K = len(class_names)
log(f"Detected classes (0..{K-1}): {class_names}")

log("Splitting train/val/test…")
X_train_all, X_test, y_train_all, y_test = train_test_split(
    df["text"], y_int, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_int
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_all, y_train_all, test_size=VAL_SIZE, random_state=RANDOM_STATE, stratify=y_train_all
)
log(f"Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}")

log(f"Vectorizing TF-IDF {NGRAMS} (max_features={MAX_FEATURES_TFIDF:,}, min_df={MIN_DF}, max_df={MAX_DF})…")
t0 = time.time()
tfidf = TfidfVectorizer(
    ngram_range=NGRAMS, sublinear_tf=True,
    min_df=MIN_DF, max_df=MAX_DF, max_features=MAX_FEATURES_TFIDF
)
Xtr = tfidf.fit_transform(X_train)
Xv  = tfidf.transform(X_val)
Xte = tfidf.transform(X_test)
log(f"Vectorized: Xtr={Xtr.shape}, Xv={Xv.shape}, Xte={Xte.shape} in {time.time()-t0:.2f}s")
log(f"Vocabulary size: {len(tfidf.get_feature_names_out()):,}")

log("Fitting LogisticRegression (saga)…")
t0 = time.time()
logit = LogisticRegression(C=6.0, class_weight="balanced", max_iter=4000,
                           n_jobs=-1, solver="saga", penalty="l2").fit(Xtr, y_train)
log(f"Done LogReg in {time.time()-t0:.2f}s")

if USE_SVC:
    log("Fitting LinearSVC + isotonic calibration (cv=3)…")
    t0 = time.time()
    svc = LinearSVC(C=2.5, class_weight="balanced", max_iter=4000, random_state=RANDOM_STATE)
    svc_cal = CalibratedClassifierCV(svc, method="isotonic", cv=3).fit(Xtr, y_train)
    log(f"Done SVC in {time.time()-t0:.2f}s")
else:
    svc_cal = None
    log("Skipped SVC calibration (USE_SVC=False).")

if USE_XGB:
    xgb_model = train_xgb_with_fallback(Xtr, y_train, Xv, y_val, K)
else:
    xgb_model = None
    log("Skipped XGBoost (USE_XGB=False).")

log("Building validation probs for ensemble…")
val_probs, names = [], []
val_probs.append(proba(logit, Xv)); names.append("logit")
if svc_cal is not None:
    val_probs.append(proba(svc_cal, Xv)); names.append("svc")
if xgb_model is not None:
    val_probs.append(proba(xgb_model, Xv)); names.append("xgb")

W = best_weights(val_probs, y_val)
log("Ensemble weights: " + str(dict(zip(names, [float(x) for x in W]))))

log("Evaluating on test…")
test_probs = [proba(logit, Xte)]
if svc_cal is not None: test_probs.append(proba(svc_cal, Xte))
if xgb_model is not None: test_probs.append(proba(xgb_model, Xte))

P = np.average(np.stack(test_probs), axis=0, weights=W)
y_pred = P.argmax(1)
acc = accuracy_score(y_test, y_pred)
f1m = f1_score(y_test, y_pred, average="macro")

log(f"Test accuracy: {acc:.4f}")
log(f"Test macro-F1: {f1m:.4f}")

eval_df = pd.DataFrame({
    "text": X_test.values,
    "y_true": y_test,
    "y_pred": y_pred,
    "p_max": P.max(axis=1),
    "char_len": X_test.astype(str).str.len(),
    "word_len": X_test.astype(str).str.split().map(len).fillna(0).astype(int),
})
for k, cname in enumerate(class_names):
    eval_df[f"p_{cname}"] = P[:,k]
eval_df["correct"] = (eval_df["y_true"] == eval_df["y_pred"]).astype(int)
eval_df.to_csv(OUT_DIR/"eval_rows.csv", index=False)

if RUN_PLOTS:
    log("Creating quick plots…")

    num_cols = eval_df.select_dtypes(include=[np.number]).columns
    corr = eval_df[num_cols].corr()
    plt.figure(); im = plt.imshow(corr, aspect="auto")
    plt.xticks(range(len(num_cols)), num_cols, rotation=90)
    plt.yticks(range(len(num_cols)), num_cols)
    plt.title("Correlation Heatmap"); plt.colorbar(im); plt.tight_layout()
    plt.savefig(OUT_DIR/"corr_heatmap.png", dpi=200); plt.close()

    roll = eval_df["correct"].rolling(window=max(50, len(eval_df)//10), min_periods=10).mean()
    series = roll.fillna(method="bfill").values
    alpha = 0.3; level = series[0]; smoothed = [level]
    for x in series[1:]:
        level = alpha*x + (1-alpha)*level; smoothed.append(level)
    forecast_h = 100; forecast = np.full(forecast_h, smoothed[-1])
    plt.figure()
    plt.plot(series, label="rolling_acc"); plt.plot(smoothed, label="smoothed")
    plt.plot(range(len(series), len(series)+forecast_h), forecast, label="forecast")
    plt.legend(); plt.title("Rolling Accuracy + Simple Forecast")
    plt.xlabel("Index"); plt.ylabel("Accuracy"); plt.tight_layout()
    plt.savefig(OUT_DIR/"trend_forecast.png", dpi=200); plt.close()

    svd50 = TruncatedSVD(n_components=40, random_state=RANDOM_STATE)
    Z50 = svd50.fit_transform(Xte)
    svd2 = TruncatedSVD(n_components=2, random_state=RANDOM_STATE)
    Z2 = svd2.fit_transform(Z50)
    km = MiniBatchKMeans(n_clusters=min(3, K), batch_size=1024, n_init=5, random_state=RANDOM_STATE)
    _ = km.fit_predict(Z2)
    plt.figure(); plt.scatter(Z2[:,0], Z2[:,1], s=6, alpha=0.6, c=y_pred, cmap="viridis")
    plt.title("MiniBatchKMeans on 2D SVD (colored by predicted class)")
    plt.xlabel("SVD-1"); plt.ylabel("SVD-2"); plt.tight_layout()
    plt.savefig(OUT_DIR/"clusters.png", dpi=200); plt.close()

log("All done")
print(f"Artifacts saved to: {OUT_DIR.resolve()}")


In [ ]:
import re, json, warnings, time
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from typing import List, Tuple

from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import MiniBatchKMeans
from sklearn.neighbors import KNeighborsClassifier
from scipy import stats

import xgboost as xgb
warnings.filterwarnings("ignore")

try:
    from google.colab import files
    print("[📁] Use the dialog to upload your CSV (must have text + label columns)…")
    up = files.upload()
    DATASET_PATH = list(up.keys())[0]
    print(f"[✔] Uploaded: {DATASET_PATH}")
except Exception:
    DATASET_PATH = "/content/data.csv"
    print(f"[ℹ] Using default path: {DATASET_PATH}")

RANDOM_STATE = 42
TEST_SIZE    = 0.20
VAL_SIZE     = 0.10

MAX_FEATURES_TFIDF = 60_000
MIN_DF = 5
MAX_DF = 0.95
NGRAMS = (1, 1)

USE_XGB      = True
USE_XGB_GPU  = True
USE_SVC      = False

XGB_MAX_ROUNDS    = 600
XGB_EARLY_STOP    = 20
XGB_LEARNING_RATE = 0.12
XGB_MAX_DEPTH     = 6
XGB_TIME_LIMIT_S  = None

OUT_DIR = Path("fast_run_artifacts")
OUT_DIR.mkdir(exist_ok=True)

def now(): return datetime.now().strftime("%H:%M:%S")
def log(msg): print(f"[{now()}] {msg}")

LIKELY_TEXT  = {'text','tweet','review','content','message','comment','body','sentence','clean_text'}
LIKELY_LABEL = {'label','labels','sentiment','target','polarity','class','category','sentiment_label'}

def _pick_col(df, cands):
    for c in df.columns:
        if str(c).strip().lower() in cands:
            return c
    return None

def load_dataset(path)->pd.DataFrame:
    t0 = time.time()
    log("Loading dataset…")
    df = pd.read_csv(path)
    df.columns = [str(c).strip().replace("\ufeff","") for c in df.columns]
    tcol = _pick_col(df, LIKELY_TEXT)  or df.columns[0]
    lcol = _pick_col(df, LIKELY_LABEL) or df.columns[1]
    df = df[[tcol, lcol]].rename(columns={tcol:"text", lcol:"label"})
    df = df.dropna(subset=["text","label"]).reset_index(drop=True)
    log(f"Loaded {len(df):,} rows (text='{tcol}', label='{lcol}') in {time.time()-t0:.2f}s")
    return df

URL = re.compile(r"https?://\S+|www\.\S+")
MENTION = re.compile(r"@\w+")
MULTI_WS = re.compile(r"\s+")

def _clean_one(s:str)->str:
    if not isinstance(s,str): return ""
    s = s.lower()
    s = URL.sub(" ", s)
    s = MENTION.sub(" ", s)
    s = s.replace("#","")
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = MULTI_WS.sub(" ", s).strip()
    return s

def clean_series(series: pd.Series, desc="Cleaning text")->pd.Series:
    log(f"{desc} (tqdm)…")
    t0 = time.time()
    tqdm.pandas(mininterval=0.5)
    out = series.progress_map(_clean_one)
    log(f"Done cleaning in {time.time()-t0:.2f}s")
    return out

def coerce_labels(y:pd.Series)->Tuple[np.ndarray, List[str]]:
    maps = [
        {"negative":-1,"neutral":0,"positive":1},
        {"neg":-1,"neu":0,"pos":1},
        {"-1":-1,"0":0,"1":1},
        {"NEGATIVE":-1,"NEUTRAL":0,"POSITIVE":1},
        {"Negative":-1,"Neutral":0,"Positive":1},
    ]
    ys = y.astype(str).str.strip()
    for m in maps:
        mm = ys.map(m)
        if mm.notna().mean() > 0.9 and set(mm.dropna().unique()) <= {-1,0,1}:
            yy = mm.fillna(0).astype(int)
            le = LabelEncoder(); enc = le.fit_transform(yy)
            classes = list(le.inverse_transform(sorted(set(enc))))
            return enc.astype(int), [str(c) for c in classes]
    le = LabelEncoder(); enc = le.fit_transform(ys)
    classes = list(le.inverse_transform(np.arange(enc.max()+1)))
    return enc.astype(int), [str(c) for c in classes]

def proba(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)
    if isinstance(model, xgb.Booster):
        return model.predict(xgb.DMatrix(X))
    raise ValueError("No probability interface")

def best_weights(p_list: List[np.ndarray], y_true: np.ndarray):
    log("Searching ensemble weights on validation…")
    grids = [np.linspace(0.5, 1.5, 5) for _ in range(len(p_list))]
    best_f1, best_w = -1, np.ones(len(p_list))/len(p_list)
    tried = 0
    if len(p_list)==2:
        for a in grids[0]:
            for b in grids[1]:
                w = np.array([a,b]); tried += 1
                mix = np.average(np.stack(p_list), axis=0, weights=w)
                f1 = f1_score(y_true, mix.argmax(1), average="macro")
                if f1>best_f1: best_f1, best_w = f1, w
    else:
        for a in grids[0]:
            for b in grids[1]:
                for c in grids[2]:
                    w = np.array([a,b,c]); tried += 1
                    mix = np.average(np.stack(p_list), axis=0, weights=w)
                    f1 = f1_score(y_true, mix.argmax(1), average="macro")
                    if f1>best_f1: best_f1, best_w = f1, w
    log(f"Grid tried={tried} combos; best macro-F1={best_f1:.4f}")
    return best_w

class TimeLimitCallback(xgb.callback.TrainingCallback):
    def __init__(self, seconds=None):
        self.seconds = seconds
        self.t0 = None
    def before_training(self, model):
        if self.seconds: import time; self.t0 = time.time()
        return model
    def after_iteration(self, model, epoch, evals_log):
        if not self.seconds: return False
        import time
        if time.time() - self.t0 > self.seconds:
            print(f"[⏱] Stopping early at iter {epoch} due to time limit ({self.seconds}s).")
            return True
        return False

def train_xgb_with_fallback(Xtr, y_train, Xv, y_val, K,
                            use_gpu=USE_XGB_GPU,
                            max_rounds=XGB_MAX_ROUNDS,
                            early_stop=XGB_EARLY_STOP,
                            lr=XGB_LEARNING_RATE,
                            depth=XGB_MAX_DEPTH,
                            time_limit_s=XGB_TIME_LIMIT_S):
    def _train_one(is_gpu: bool):
        dtr = xgb.DMatrix(Xtr, label=y_train)
        dv  = xgb.DMatrix(Xv,  label=y_val)
        params = {
            "objective": "multi:softprob",
            "num_class": K,
            "eta": lr,
            "max_depth": depth,
            "subsample": 0.8,
            "colsample_bytree": 0.8,
            "reg_lambda": 1.2,
            "eval_metric": ["mlogloss", "merror"],
            "seed": RANDOM_STATE,
            "tree_method": "gpu_hist" if is_gpu else "hist",
            "predictor": "gpu_predictor" if is_gpu else "auto",
            "nthread": -1,
        }
        ev = {}
        callbacks = []
        if time_limit_s is not None:
            callbacks.append(TimeLimitCallback(time_limit_s))
        log(f"Fitting XGBoost on {'GPU' if is_gpu else 'CPU'} (fast schedule, early stop)…")
        booster = xgb.train(
            params, dtr, num_boost_round=max_rounds,
            evals=[(dv, "val")],
            early_stopping_rounds=early_stop,
            verbose_eval=25,
            evals_result=ev,
            callbacks=callbacks
        )
        booster._evals_result = ev
        log(f"XGBoost best_iteration={booster.best_iteration} on {'GPU' if is_gpu else 'CPU'}")
        return booster

    if use_gpu:
        try:
            return _train_one(is_gpu=True)
        except xgb.core.XGBoostError as e:
            msg = str(e)
            log(f"[WARN] GPU training failed: {msg.strip().splitlines()[-1]}")
            log("Retrying on CPU ('hist')…")
            return _train_one(is_gpu=False)
    else:
        return _train_one(is_gpu=False)

def save_curve_from_xgb(booster, out_prefix="xgb"):
    try:
        ev = booster._evals_result
        val_logloss = ev["val"]["mlogloss"]
        val_merror  = ev["val"]["merror"]

        plt.figure()
        plt.plot(val_logloss)
        plt.title("XGBoost Validation mlogloss")
        plt.xlabel("Iteration"); plt.ylabel("mlogloss")
        plt.tight_layout(); plt.savefig(OUT_DIR/f"{out_prefix}_loss_curve.png", dpi=200); plt.close()

        plt.figure()
        plt.plot(1 - np.array(val_merror, dtype=float))
        plt.title("XGBoost Validation Accuracy (1 - merror)")
        plt.xlabel("Iteration"); plt.ylabel("Accuracy")
        plt.tight_layout(); plt.savefig(OUT_DIR/f"{out_prefix}_accuracy_curve.png", dpi=200); plt.close()
    except Exception as e:
        log(f"[WARN] Could not save XGB curves: {e}")

def plot_corr_heatmap_numbers(df_eval, fname="corr_heatmap_numbers.png"):
    num_cols = df_eval.select_dtypes(include=[np.number]).columns
    corr = df_eval[num_cols].corr()
    plt.figure(figsize=(max(6, 0.5*len(num_cols)), max(5, 0.5*len(num_cols))))
    im = plt.imshow(corr, aspect="auto")
    plt.xticks(range(len(num_cols)), num_cols, rotation=90)
    plt.yticks(range(len(num_cols)), num_cols)

    for i in range(len(num_cols)):
        for j in range(len(num_cols)):
            plt.text(j, i, f"{corr.iloc[i,j]:.2f}", ha="center", va="center", fontsize=8)
    plt.title("Correlation Heatmap (with values)")
    plt.colorbar(im)
    plt.tight_layout()
    plt.savefig(OUT_DIR/fname, dpi=200); plt.close()

def clustering_graph(X_sparse_test, y_pred, fname="clusters.png"):
    svd50 = TruncatedSVD(n_components=40, random_state=RANDOM_STATE)
    Z50 = svd50.fit_transform(X_sparse_test)
    svd2 = TruncatedSVD(n_components=2, random_state=RANDOM_STATE)
    Z2 = svd2.fit_transform(Z50)
    km = MiniBatchKMeans(n_clusters=3, batch_size=1024, n_init=5, random_state=RANDOM_STATE)
    _ = km.fit_predict(Z2)
    plt.figure()
    plt.scatter(Z2[:,0], Z2[:,1], s=6, alpha=0.6, c=y_pred, cmap="viridis")
    plt.title("MiniBatchKMeans on 2D SVD (colored by predicted class)")
    plt.xlabel("SVD-1"); plt.ylabel("SVD-2")
    plt.tight_layout(); plt.savefig(OUT_DIR/fname, dpi=200); plt.close()
    return Z2

def linear_regression_graph(df_eval, fname="linear_regression.png"):
    X = df_eval[["char_len","word_len"]].values
    y = df_eval["p_max"].values

    n = len(y); idx = np.arange(n); rs = np.random.RandomState(42); rs.shuffle(idx)
    cut = int(0.8*n); tr, te = idx[:cut], idx[cut:]
    lr = LinearRegression().fit(X[tr], y[tr])
    y_pred = lr.predict(X[te])

    plt.figure()
    plt.scatter(y[te], y_pred, s=8, alpha=0.6)
    plt.plot([0,1],[0,1])
    plt.title(f"Linear Regression: predict confidence from lengths (R^2={lr.score(X[te], y[te]):.3f})")
    plt.xlabel("Actual p_max"); plt.ylabel("Predicted p_max")
    plt.tight_layout(); plt.savefig(OUT_DIR/fname, dpi=200); plt.close()

def bar_graphs(df_eval, class_names, y_true, y_pred):
    vals, counts = np.unique(y_true, return_counts=True)
    plt.figure()
    plt.bar([str(class_names[v]) for v in vals], counts)
    plt.title("Class Distribution (Test)"); plt.xlabel("Class"); plt.ylabel("Count")
    plt.tight_layout(); plt.savefig(OUT_DIR/"bar_class_distribution.png", dpi=200); plt.close()

    cm = confusion_matrix(y_true, y_pred, labels=vals)
    per_class_acc = cm.diagonal() / cm.sum(axis=1).clip(min=1)
    plt.figure()
    plt.bar([str(class_names[v]) for v in vals], per_class_acc)
    plt.title("Per-class Accuracy (Test)"); plt.xlabel("Class"); plt.ylabel("Accuracy")
    plt.ylim(0,1)
    plt.tight_layout(); plt.savefig(OUT_DIR/"bar_per_class_accuracy.png", dpi=200); plt.close()

def knn_graph(Z2, y_true, fname="knn_graph.png", k=5):
    knn = KNeighborsClassifier(n_neighbors=k, weights="distance")
    knn.fit(Z2, y_true)
    y_knn = knn.predict(Z2)
    plt.figure()
    plt.scatter(Z2[:,0], Z2[:,1], s=6, alpha=0.6, c=y_knn, cmap="viridis")
    plt.title(f"KNN (k={k}) Predictions on 2D SVD space")
    plt.xlabel("SVD-1"); plt.ylabel("SVD-2")
    plt.tight_layout(); plt.savefig(OUT_DIR/fname, dpi=200); plt.close()

def anova_graph(df_eval, class_names, fname="anova.png"):
    groups = []
    labels = []
    for cls_idx, cls_name in enumerate(class_names):
        g = df_eval.loc[df_eval["y_true"]==cls_idx, "p_max"].values
        if len(g) > 1:
            groups.append(g); labels.append(str(cls_name))
    if len(groups) >= 2:
        F, p = stats.f_oneway(*groups)
    else:
        F, p = np.nan, np.nan

    means = [np.mean(g) for g in groups]
    stds  = [np.std(g) for g in groups]
    plt.figure()
    plt.bar(labels, means, yerr=stds, capsize=4)
    plt.title(f"ANOVA: p_max across classes (F={F:.3f}, p={p:.3e})")
    plt.xlabel("Class"); plt.ylabel("Mean p_max")
    plt.ylim(0,1)
    plt.tight_layout(); plt.savefig(OUT_DIR/fname, dpi=200); plt.close()

    with open(OUT_DIR/"anova_stats.json","w") as f:
        json.dump({"F": float(F), "p_value": float(p),
                   "means": dict(zip(labels, [float(m) for m in means])),
                   "stds": dict(zip(labels, [float(s) for s in stds]))}, f, indent=2)

df = load_dataset(DATASET_PATH)
df["text"] = clean_series(df["text"])

y_int, class_names = coerce_labels(df["label"])
K = len(class_names)
log(f"Detected classes (0..{K-1}): {class_names}")

log("Splitting train/val/test…")
X_train_all, X_test, y_train_all, y_test = train_test_split(
    df["text"], y_int, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_int
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_all, y_train_all, test_size=VAL_SIZE, random_state=RANDOM_STATE, stratify=y_train_all
)
log(f"Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}")

log(f"Vectorizing TF-IDF {NGRAMS} (max_features={MAX_FEATURES_TFIDF:,}, min_df={MIN_DF}, max_df={MAX_DF})…")
t0 = time.time()
tfidf = TfidfVectorizer(
    ngram_range=NGRAMS, sublinear_tf=True,
    min_df=MIN_DF, max_df=MAX_DF, max_features=MAX_FEATURES_TFIDF
)
Xtr = tfidf.fit_transform(X_train)
Xv  = tfidf.transform(X_val)
Xte = tfidf.transform(X_test)
log(f"Vectorized: Xtr={Xtr.shape}, Xv={Xv.shape}, Xte={Xte.shape} in {time.time()-t0:.2f}s")
log(f"Vocabulary size: {len(tfidf.get_feature_names_out()):,}")

log("Fitting LogisticRegression (saga)…")
t0 = time.time()
logit = LogisticRegression(C=6.0, class_weight="balanced", max_iter=4000,
                           n_jobs=-1, solver="saga", penalty="l2").fit(Xtr, y_train)
log(f"Done LogReg in {time.time()-t0:.2f}s")

if USE_SVC:
    log("Fitting LinearSVC + isotonic calibration (cv=3)…")
    t0 = time.time()
    svc = LinearSVC(C=2.5, class_weight="balanced", max_iter=4000, random_state=RANDOM_STATE)
    svc_cal = CalibratedClassifierCV(svc, method="isotonic", cv=3).fit(Xtr, y_train)
    log(f"Done SVC in {time.time()-t0:.2f}s")
else:
    svc_cal = None
    log("Skipped SVC calibration (USE_SVC=False).")

if USE_XGB:
    xgb_model = train_xgb_with_fallback(Xtr, y_train, Xv, y_val, K)
else:
    xgb_model = None
    log("Skipped XGBoost (USE_XGB=False).")

log("Building validation probs for ensemble…")
val_probs, names = [], []
val_probs.append(proba(logit, Xv)); names.append("logit")
if svc_cal is not None:
    val_probs.append(proba(svc_cal, Xv)); names.append("svc")
if xgb_model is not None:
    val_probs.append(proba(xgb_model, Xv)); names.append("xgb")

W = best_weights(val_probs, y_val)
log("Ensemble weights: " + str(dict(zip(names, [float(x) for x in W]))))

log("Evaluating on test…")
test_probs = [proba(logit, Xte)]
if svc_cal is not None: test_probs.append(proba(svc_cal, Xte))
if xgb_model is not None: test_probs.append(proba(xgb_model, Xte))

P = np.average(np.stack(test_probs), axis=0, weights=W)
y_pred = P.argmax(1)
acc = accuracy_score(y_test, y_pred)
f1m = f1_score(y_test, y_pred, average="macro")
log(f"Test accuracy: {acc:.4f}")
log(f"Test macro-F1: {f1m:.4f}")

eval_df = pd.DataFrame({
    "text": X_test.values,
    "y_true": y_test,
    "y_pred": y_pred,
    "p_max": P.max(axis=1),
    "char_len": X_test.astype(str).str.len(),
    "word_len": X_test.astype(str).str.split().map(len).fillna(0).astype(int),
})
for k, cname in enumerate(class_names):
    eval_df[f"p_{cname}"] = P[:,k]
eval_df["correct"] = (eval_df["y_true"] == eval_df["y_pred"]).astype(int)
eval_df.to_csv(OUT_DIR/"eval_rows.csv", index=False)

log("Saving requested plots…")
if xgb_model is not None:
    save_curve_from_xgb(xgb_model, out_prefix="xgb")

plot_corr_heatmap_numbers(eval_df, fname="corr_heatmap_numbers.png")

Z2 = clustering_graph(Xte, y_pred, fname="clusters.png")

linear_regression_graph(eval_df, fname="linear_regression.png")

bar_graphs(eval_df, class_names, y_test, y_pred)

knn_graph(Z2, y_test, fname="knn_graph.png", k=5)

anova_graph(eval_df, class_names, fname="anova.png")

rpt = classification_report(y_test, y_pred, target_names=[str(c) for c in class_names], output_dict=True)
pd.DataFrame(rpt).transpose().to_csv(OUT_DIR/"classification_report.csv")

log("All done")
print(f"Artifacts saved to: {OUT_DIR.resolve()}")
print("Files generated:")
for p in sorted(OUT_DIR.glob("*")):
    print(" -", p.name)